# Pix2Struct AI2D-base — DIMER E2E diagram multiple-choice fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/pix2struct-ai2d-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/pix2struct-ai2d-pipeline/blob/main/tutorials/pix2struct_ai2d_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google%2Fpix2struct--ai2d--base-ffcc4d?style=flat)](https://huggingface.co/google/pix2struct-ai2d-base) [![Upstream](https://img.shields.io/badge/Upstream-google--research%2Fpix2struct-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/pix2struct) [![arXiv](https://img.shields.io/badge/arXiv-2210.03347-b31b1b.svg)](https://arxiv.org/abs/2210.03347)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** diagram multiple-choice question answering and bounded supervised fine-tuning of the answer decoder's last blocks on a diagram/question/options dataset, using the pinned `google/pix2struct-ai2d-base` weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/pix2struct_ai2d_pipeline/`, at revision `e575bd6b4c97`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `0d6b2606efe05c77c1d0670647740802a9e68eef` (~569 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned `google/pix2struct-ai2d-base` snapshot (a 565 MB `model.safetensors`), downloads one digest-pinned parquet shard of AI2D test questions from the Hugging Face Hub (62 MB, no credential, refused on any size or SHA-256 mismatch), cuts a seeded subset of whole diagrams into training, validation and test questions so no diagram is shared, answers ten authored questions on a drawn plant diagram through the inference contract with an input manifest and a rejection probe, scores the frozen model on the test questions beside chance and two non-neural baselines, runs a bounded fine-tuning of the answer decoder's last blocks with validation-accuracy epoch selection, scores the held-out questions again per category, re-answers the drawn diagram with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify answer parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). A CUDA runtime is used automatically when present; the CPU path works but is slow (every question renders its own header and is encoded at up to 2,048 patches), and the timings of the first clean run are recorded in `docs/release-verification.md`.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to upload one zip holding a `records.jsonl` (or `records.json`) of `{{id, image, question, options, answer}}` objects — `image` a file name inside the zip, `options` 2–6 distinct strings, `answer` the zero-based index of the correct option, optional `image_id` and `category` — beside the image files. They pass through the same validation, seeded diagram-disjoint split, baselines, fine-tuning, held-out evaluation, artifact export and reload-parity cells as the AI2D sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

`google/pix2struct-ai2d-base` is the Pix2Struct model of Lee et al. (2023) — a ViT-style image encoder over variable-resolution 16×16 patches (up to 2,048 per image) and a 12-layer text decoder that cross-attends to them; 282,285,696 parameters, pretrained by parsing masked web screenshots into simplified HTML and fine-tuned on AI2D, a set of school-science diagrams with multiple-choice questions — published under the **Apache-2.0** licence. The question and its numbered options are **rendered as a text header above the diagram** (the Pix2Struct convention for visual question answering, `<question> (1) <a> (2) <b> …`, in Pillow's bundled font so no font is downloaded), the composite is encoded, and the decoder generates the answer text with greedy decoding under a caller-owned `max_new_tokens` budget; the carried module matches that text to the options by normalised exact match. **No score exists**: the answer is generated text with no probability and no correctness signal, and matching an option is **not evidence that it is the right one**.

What this notebook adds to inference is **adaptation with labelled questions**. The dataset is real and from the checkpoint's own domain: AI2D test questions (Kembhavi et al., ECCV 2016) as mirrored on the Hub — the checkpoint was fine-tuned on AI2D's *training* questions, so these diagrams and questions are unseen but not out of distribution, and the honest question is a narrow one: does a bounded adaptation of the answer decoder's last blocks on a few hundred more in-domain questions move held-out accuracy at all, and on which kind of question? About a third of AI2D questions ask about lettered diagram labels (`A`, `B`, `C`, `D` as options), the rest have text options, so the per-category breakdown (`letter-label` / `text-option`) is part of the reading. The notebook downloads **one pinned parquet shard** (62 MB, SHA-256 pinned in the carried module) and draws a seeded subset of whole diagrams from it. Accuracy counts an answer that matches no option as wrong, and three references frame it: **chance** (the mean of 1/options), the **position-prior** baseline (always the option position most often correct in training) and the **longest-option** baseline — two systems that never look at the diagram. Nothing here is a quality claim about your diagrams: it is one seeded split of one shard.

**Weight-format note:** the pinned revision ships the model as SafeTensors (`model.safetensors`, stored in bfloat16 and upcast to float32 at load, digest-pinned in the manifest); the processor is the VQA variant, which renders the header. Section 3 stages and digest-verifies the snapshot before the processor or the model is constructed.

**Learning objectives:** install the pinned runtime; read what the carried pipeline, metrics and dataset modules guarantee; stage and digest-verify the immutable upstream snapshot; download a digest-pinned shard of labelled diagram questions, validate it and split it by diagram without leakage; answer through the public API on a drawn diagram with authored questions and read `answer`, `choice_index`, `new_tokens` and `truncated` correctly (generated text matched to an option or to none, no score); score the frozen model's accuracy beside chance and two non-neural baselines and read the `letter-label` / `text-option` breakdown; run a bounded fine-tuning with explicit hyperparameters and validation-based epoch selection; evaluate on a diagram-disjoint test split; re-answer a drawing from a different image family with the adapted model; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** Free-form or open-ended answers (the contract matches the generated text to the supplied options and reports `null` otherwise), document, chart or scene question answering (separate checkpoints), reading a diagram's text back as a transcript, answer localisation, batch throughput, sampling or beam search (the notebook decodes greedily for reproducibility), evaluation on the full AI2D benchmark (only a seeded subset of one shard is scored here), fine-tuning of the image encoder, the embeddings or the output projection, training on diagrams that are not the pinned sample or your own uploads, and any claim that an AI2D split stands in for your diagrams. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available; a GPU runtime is recommended for Sections 6–8. Every question renders its own header above its diagram and is encoded at up to 2,048 patches, so each answer costs seconds on CPU. The pinned `torch==2.14.0` install and the 565 MB checkpoint are the large downloads of the run; the question shard adds 62 MB.
- **Knowledge:** basic Python and PIL; what an encoder–decoder model's generated tokens are; what accuracy against chance and against baselines that never see the image does and does not show; why a confident answer is not a correct one.
- **Data contract:** records are `{{id, image, question, options, answer}}` — an image file decodable by Pillow with sides between `MIN_IMAGE_SIDE` (16) and `MAX_IMAGE_SIDE` (4096) px, a non-empty question of at most `MAX_QUESTION_CHARS` (256) characters, `MIN_OPTIONS`..`MAX_OPTIONS` (2..6) distinct options of at most `MAX_OPTION_CHARS` (64) characters, and `answer` the zero-based index of the correct option; optional `image_id` groups questions on the same diagram (BYOD defaults it to the image file name) and optional `category` labels the breakdown. Ids match `[A-Za-z0-9_.:-]{{1,64}}` and are unique; a dataset needs 8..5,000 records; every question on the same diagram lands in the same split so a test diagram is never trained on. BYOD accepts one zip of images plus a `records.jsonl` / `records.json` in that shape.
- **Validation is structural, not semantic:** every diagram is opened and decoded and every question checked against the same ceilings `answer` applies, but nothing checks that the marked answer is right — a mislabelled question is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path downloads one object from the Hub dataset repository `lmms-lab-encoder/ai2d` at the immutable revision `c83a9b96…` (`data/test-00000-of-00002.parquet`, 62,292,686 bytes) and refuses it unless its size and SHA-256 match the pins carried in `samples.py`; the diagrams are written to the cache under their own content digest. The Hub mirror declares no licence; AI2D is published by the Allen Institute for AI (Kembhavi et al., 2016) — check its terms before redistributing the diagrams or an adapter trained on them.
- **External access:** the Hugging Face Hub only, to fetch the pinned `google/pix2struct-ai2d-base` snapshot (~569 MB in total) at revision `0d6b2606efe0…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
    'pyarrow==25.0.1',
]
NOTEBOOK_SOURCE = {
    'repository': 'pix2struct-ai2d-pipeline',
    'repository_revision': 'e575bd6b4c97a3364ed447ffd270a37311921970',
    'embedded_module': 'src/pix2struct_ai2d_pipeline/pipeline.py',
    'embedded_modules': ['src/pix2struct_ai2d_pipeline/metrics.py', 'src/pix2struct_ai2d_pipeline/pipeline.py', 'src/pix2struct_ai2d_pipeline/samples.py'],
    'module_sha256': '02283ddc9f60acc3def2b4d07c4e6720c2fcfea00c45997c1edf04b2b6d0b4ab',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/pix2struct_ai2d_pipeline/` @ `e575bd6b4c97`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/pix2struct_ai2d_pipeline/metrics.py`

In [ ]:
"""Multiple-choice scoring for diagram question answering: accuracy with unmatched answers counted wrong, the
unmatched rate, the chance level, a per-category breakdown, and two non-neural baselines that answer without
looking at the diagram.

An answer is correct when the option its generated text matches (normalised exact match, `match_option`)
is the record's correct option; an answer that matches no option is wrong. `chance` is the mean of
1 / n_options over the scored questions — what uniform guessing would score. The **position-prior baseline**
answers every test question with the option position most often correct in the training split (clamped to
the question's option count); the **longest-option baseline** answers with the longest option text (the
first on ties). Both see the question text or the option layout only, never the diagram, so a system that
does not beat them has not shown it reads the diagram.
"""

from __future__ import annotations

from collections import Counter, defaultdict
from collections.abc import Mapping, Sequence
from typing import Any

METRIC_DEFINITIONS: dict[str, str] = {
    "accuracy": (
        "fraction of questions whose matched option is the correct one; unmatched answers count as wrong"
    ),
    "unmatched_rate": "fraction of questions whose generated text matches none of the options",
    "chance": "mean of 1 / n_options over the scored questions (uniform guessing)",
    "by_category": "accuracy per record category (`letter-label` / `text-option` in the sample)",
}


def mcq_metrics(
    predicted: Sequence[int | None],
    correct: Sequence[int],
    n_options: Sequence[int],
    *,
    categories: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Accuracy, unmatched rate, chance and per-category accuracy for aligned predictions."""
    n = len(predicted)
    if n == 0 or len(correct) != n or len(n_options) != n:
        raise ValueError("predicted, correct and n_options must be non-empty and aligned")
    if categories is not None and len(categories) != n:
        raise ValueError("categories must align with the predictions")
    for gold, count in zip(correct, n_options, strict=True):
        if isinstance(gold, bool) or not isinstance(gold, int) or not 0 <= gold < count:
            raise ValueError("each correct index must be a valid zero-based option index")
    hits = [p is not None and p == g for p, g in zip(predicted, correct, strict=True)]
    groups: dict[str, list[bool]] = defaultdict(list)
    for hit, category in zip(hits, categories or ["all"] * n, strict=True):
        groups[str(category)].append(hit)
    return {
        "n": n,
        "accuracy": sum(hits) / n,
        "unmatched_rate": sum(p is None for p in predicted) / n,
        "chance": sum(1.0 / c for c in n_options) / n,
        "by_category": {
            name: {"n": len(values), "accuracy": sum(values) / len(values)}
            for name, values in sorted(groups.items())
        },
        "definitions": dict(METRIC_DEFINITIONS),
    }


def _score(
    name: str, note: str, predicted: list[int], test: Sequence[Mapping[str, Any]]
) -> dict[str, Any]:
    metrics = mcq_metrics(
        predicted,
        [int(r["answer"]) for r in test],
        [len(r["options"]) for r in test],
        categories=[str(r.get("category", "other")) for r in test],
    )
    return {**metrics, "baseline": name, "note": note}


def position_prior_baseline(
    train: Sequence[Mapping[str, Any]], test: Sequence[Mapping[str, Any]]
) -> dict[str, Any]:
    """Answer every test question with the option position most often correct in `train`."""
    if not train or not test:
        raise ValueError("train and test must be non-empty")
    counts = Counter(int(r["answer"]) for r in train)
    position = min(counts, key=lambda k: (-counts[k], k))
    predicted = [min(position, len(r["options"]) - 1) for r in test]
    note = f"always option {position + 1} (most frequent correct position in training)"
    return {**_score("position-prior", note, predicted, test), "position": position}


def longest_option_baseline(test: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Answer every test question with its longest option text (the first on ties)."""
    if not test:
        raise ValueError("test must be non-empty")
    predicted = [max(range(len(r["options"])), key=lambda k, r=r: (len(r["options"][k]), -k)) for r in test]
    note = "the longest option text, without looking at the diagram"
    return _score("longest-option", note, predicted, test)

**Module 2/3:** `src/pix2struct_ai2d_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""Diagram multiple-choice question answering with the pinned ``google/pix2struct-ai2d-base`` checkpoint.

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the Pix2Struct architecture comes from the pinned ``transformers`` release,
the weights are SafeTensors, and no model-repository code is executed. The question and its numbered
options are rendered as a text header on top of the diagram (the Pix2Struct VQA input convention) with
Pillow's bundled font, so no font is fetched from the Hub at inference time; the generated answer text
is matched to the options by normalised exact match.

The adaptation contract (`evaluate`, `adapt`, `save_artifact`, `load_artifact`, `from_artifact`) fine-tunes
the answer decoder's last blocks on validated multiple-choice records with the correct option's text as the
target, selects the epoch on validation accuracy and exports the trained tensors as a safetensors adapter
bound to the pinned base.
"""

from __future__ import annotations

import hashlib
import json
import math
import re
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from PIL import Image, ImageFont

MODEL_ID = "google/pix2struct-ai2d-base"
MODEL_REVISION = "0d6b2606efe05c77c1d0670647740802a9e68eef"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "pix2struct-ai2d-base"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHT_FILE = "model.safetensors"
WEIGHT_SHA256 = "652c92f8b995d9fc5ea46350c854fd88767bf36c391552f126f6e7487b0f27a3"
PARAMETER_COUNT = 282_285_696  # 18,879,744 of them train by default (2 decoder blocks + final norm)
DECODER_LAYERS = 12
DEFAULT_TRAINABLE_DECODER_LAYERS = 2
# Evaluation bounds: a split larger than MAX_EVAL_RECORDS is refused (answering is one encoder pass per
# question); below MIN_SCORED_RECORDS the verdict says the sample is small.
MAX_EVAL_RECORDS = 2_000
MIN_SCORED_RECORDS = 50
ARTIFACT_FORMAT = "org.valcorza.pix2struct-ai2d-base.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"

# Generation ceilings. AI2D answers are one option's text (the checkpoint's text_config max_length is
# 20); the default leaves room for a long option, the ceiling bounds runaway generation.
MAX_NEW_TOKENS = 64
DEFAULT_MAX_NEW_TOKENS = 16
DECODING = "greedy"
# Prompt ceilings. The question and the numbered options are rendered as header lines (wrapped at 80
# characters by the processor) above the diagram; a long prompt shrinks the diagram's share of the
# patch budget. AI2D questions carry four options; the ceiling allows a few more.
MAX_QUESTION_CHARS = 256
MAX_OPTION_CHARS = 64
MIN_OPTIONS = 2
MAX_OPTIONS = 6
# Input ceilings. The processor extracts at most MAX_PATCHES 16x16 patches (preprocessor_config.json)
# after scaling the image to fill that budget, so pixel count only guards memory during resizing.
MAX_PATCHES = 2048
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
_PUNCT_RE = re.compile(r"[^\w\s]")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def header_font_bytes() -> bytes:
    """Pillow's bundled Aileron Regular (CC0) as TrueType bytes: the header font for the rendered question.

    The upstream image processor otherwise fetches ``ybelkada/fonts/Arial.TTF`` from the Hub at
    inference time — an unpinned, unlisted download of a proprietary font. The bundled subset covers
    the printable ASCII range, which is what a question is expected to use.
    """
    font = ImageFont.load_default(size=36)
    data = getattr(font, "font_bytes", None)
    if not data:
        raise RuntimeError("Pillow's bundled TrueType font is unavailable (FreeType support missing)")
    return bytes(data)


def normalize_answer(text: str) -> str:
    """Normalisation for option matching: lower-case, punctuation removed, whitespace collapsed."""
    return " ".join(_PUNCT_RE.sub(" ", text.lower()).split())


def format_prompt(question: str, options: Sequence[str]) -> str:
    """The pinned README's AI2D prompt convention: ``<question> (1) <a> (2) <b> ...``."""
    return " ".join([question] + [f"({index}) {option}" for index, option in enumerate(options, start=1)])


def match_option(answer: str, options: Sequence[str]) -> int | None:
    """Zero-based index of the option the normalised answer equals, else ``None`` (no fuzzy match)."""
    pred = normalize_answer(answer)
    for index, option in enumerate(options):
        if pred == normalize_answer(option):
            return index
    return None


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one diagram image as PIL.Image.Image (any mode, converted to RGB) plus one question string and "
        "MIN_OPTIONS..MAX_OPTIONS answer options"
    ),
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "question_chars": [1, MAX_QUESTION_CHARS],
    "options": [MIN_OPTIONS, MAX_OPTIONS],
    "option_chars": [1, MAX_OPTION_CHARS],
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "decoding": f"{DECODING} (do_sample=False), deterministic on a fixed device and dtype",
    "preprocessing": (
        "the question and the numbered options are rendered as a black-on-white header (Pillow's bundled "
        "font, wrapped at 80 characters) above the diagram; the composite is scaled to fill at most "
        "MAX_PATCHES 16x16 patches (aspect ratio preserved), normalised per image and flattened into patch "
        "tokens with row/column positions; the decoder generates the answer text, which is matched to the "
        "options by normalised exact match"
    ),
    "output": "one answer string (the model's decoded text) plus the matched option index or null; no score",
}


def _check_inputs(
    image: Any, question: Any, options: Any, max_new_tokens: Any
) -> tuple[Image.Image, str, list[str], int]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``answer`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    if not isinstance(question, str):
        raise TypeError("question must be a str")
    checked_question = " ".join(question.split())
    if not checked_question:
        raise ValueError("question must contain at least one non-whitespace character")
    if len(checked_question) > MAX_QUESTION_CHARS:
        raise ValueError(
            f"question has {len(checked_question)} chars > MAX_QUESTION_CHARS {MAX_QUESTION_CHARS}"
        )
    if isinstance(options, str) or not isinstance(options, Sequence):
        raise TypeError("options must be a sequence of str")
    if not MIN_OPTIONS <= len(options) <= MAX_OPTIONS:
        raise ValueError(f"options must have MIN_OPTIONS={MIN_OPTIONS}..MAX_OPTIONS={MAX_OPTIONS} entries")
    checked_options = []
    for option in options:
        if not isinstance(option, str):
            raise TypeError("each option must be a str")
        checked = " ".join(option.split())
        if not checked:
            raise ValueError("each option must contain at least one non-whitespace character")
        if len(checked) > MAX_OPTION_CHARS:
            raise ValueError(f"option has {len(checked)} chars > MAX_OPTION_CHARS {MAX_OPTION_CHARS}")
        checked_options.append(checked)
    if len({normalize_answer(option) for option in checked_options}) != len(checked_options):
        raise ValueError("options must be distinct after normalisation")
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    return rgb, checked_question, checked_options, max_new_tokens


def validate_inputs(
    image: Image.Image,
    questions: Sequence[Mapping[str, Any]],
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    ``questions`` holds mappings with ``question`` and ``options``; each is checked exactly as
    ``answer`` would check it. Rejection is reported by raising, and a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    if isinstance(questions, (str, Mapping)) or not isinstance(questions, Sequence) or not questions:
        raise TypeError("questions must be a non-empty sequence of {question, options} mappings")
    checked = []
    for entry in questions:
        if not isinstance(entry, Mapping) or "question" not in entry or "options" not in entry:
            raise TypeError("each questions entry must be a mapping with 'question' and 'options'")
        _, question, options, _ = _check_inputs(image, entry["question"], entry["options"], max_new_tokens)
        checked.append({"question": question, "options": options, "prompt": format_prompt(question, options)})
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (answer takes one diagram)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": names[0] if names else "image-0", "mode": image.mode, "size": list(image.size)}],
        "questions": checked,
        "generation": {"max_new_tokens": int(max_new_tokens), "do_sample": False, "decoding": DECODING},
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Sequence[Mapping[str, Any]],
    correct: Sequence[int] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``correct`` (the zero-based index of the correct option per result, in order) the report
    carries the ``accuracy`` (matched option equals the correct index; an unmatched answer counts as
    wrong), the chance baseline, the rate of unmatched answers and one per-question entry, verdict
    ``sample-sanity``; without it the report is ``not-measurable`` and says what labelled data would
    make the task measurable.
    """
    if not results:
        raise ValueError("results must contain at least one answer result")
    base = {
        "task": "diagram image + multiple-choice question -> option text (AI2D-style diagram QA)",
        "score_semantics": (
            "the answer is generated text and carries no score, probability or correctness signal; the "
            "option index is a normalised exact match of that text against the options and is null when "
            "the text matches none of them. Greedy decoding makes the output reproducible on a fixed device "
            "and dtype, a reproducibility property, not a quality one"
        ),
        "sample_kind": sample_kind,
        "n_questions": len(results),
        "truncated": [bool(result.get("truncated")) for result in results],
        "unmatched": [result.get("choice_index") is None for result in results],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if correct is None:
        return {
            **base,
            "metrics": [],
            "baselines": [],
            "verdict": "not-measurable",
            "reason": "no correct-option labels were supplied for the evaluated questions",
            "needs": (
                "multiple-choice question/answer pairs on diagrams from the deployment domain (AI2D-style "
                "annotations with the correct option marked) scored with accuracy; no such labelled set "
                "ships with this repository"
            ),
        }
    if len(correct) != len(results):
        raise ValueError(f"correct has {len(correct)} entries for {len(results)} results")
    per_question = []
    for result, gold in zip(results, correct, strict=True):
        options = list(result.get("options") or [])
        if isinstance(gold, bool) or not isinstance(gold, int) or not 0 <= gold < len(options):
            raise ValueError("each correct entry must be a valid zero-based option index for its result")
        predicted = result.get("choice_index")
        per_question.append(
            {
                "question": result.get("question"),
                "prediction": result.get("answer"),
                "choice_index": predicted,
                "correct_index": gold,
                "correct_option": options[gold],
                "correct": predicted == gold,
            }
        )
    n_options = [len(result.get("options") or []) for result in results]
    chance = sum(1.0 / n for n in n_options) / len(n_options)
    return {
        **base,
        "metrics": [
            {
                "id": "accuracy",
                "value": sum(entry["correct"] for entry in per_question) / len(per_question),
                "normalisation": "answer text lower-cased, punctuation removed, whitespace collapsed; "
                "exact match against the options; unmatched counts as wrong",
                "estimation": f"{len(per_question)} question(s) on one diagram, no dispersion estimate",
            },
            {
                "id": "unmatched_rate",
                "value": sum(entry["choice_index"] is None for entry in per_question) / len(per_question),
                "estimation": f"{len(per_question)} question(s), answers matching no option",
            },
        ],
        "baselines": [{"id": "chance", "value": chance, "note": "mean of 1/n_options over the questions"}],
        "per_question": per_question,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(per_question)} authored question(s) on one tutorial diagram you drew yourself; plumbing "
            "evidence, not an AI2D benchmark"
        ),
        "needs": (
            "a labelled multiple-choice set on diagrams from the deployment domain (subject, drawing style, "
            "label density) for any accuracy claim; the AI2D benchmark is not bundled"
        ),
    }




@dataclass
class Pix2StructAI2DPipeline:
    """``_runner(image, prompt, max_new_tokens)`` returns ``{"answer": str, "new_tokens": int}``;
    injectable so the offline tests run without the model."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"
    adapter: dict[str, Any] | None = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)
    _processor: Any = field(default=None, repr=False)
    _font_bytes: bytes | None = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Pix2StructAI2DPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        header_font_bytes()
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import Pix2StructForConditionalGeneration, Pix2StructProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = Pix2StructProcessor.from_pretrained(location, **common)
        if not getattr(processor.image_processor, "is_vqa", False):
            raise RuntimeError("snapshot image processor is not the VQA variant (is_vqa=False); refusing")
        # The checkpoint is stored in bfloat16; it is upcast to float32 for CPU inference and training.
        model = Pix2StructForConditionalGeneration.from_pretrained(location, dtype=torch.float32, **common)
        return cls._from_model(model, processor, resolved_device, source)

    @classmethod
    def _from_model(cls, model: Any, processor: Any, device: str, source: str) -> Pix2StructAI2DPipeline:
        """Wrap a constructed model and VQA processor (every parameter frozen, eval mode) in a pipeline; the
        offline tests use it with a small randomly initialised Pix2Struct model."""
        import torch

        font_bytes = header_font_bytes()
        model = model.eval().to(device)
        for param in model.parameters():
            param.requires_grad_(False)

        def runner(image: Image.Image, prompt: str, max_new_tokens: int) -> dict[str, Any]:
            # The image processor is called directly: Pix2StructProcessor.__call__ drops the
            # font_bytes kwarg, and font_bytes is what replaces the default Hub font download
            # (see header_font_bytes). The VQA processor renders the prompt as the header.
            inputs = processor.image_processor(
                image, header_text=prompt, return_tensors="pt", font_bytes=font_bytes
            ).to(device)
            with torch.inference_mode():
                generated = model.generate(
                    **inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1
                )
            # Encoder-decoder: the output holds only decoder tokens (decoder_start + answer + eos).
            answer_ids = generated[0]
            decoded = processor.tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
            return {"answer": decoded, "new_tokens": int(answer_ids.shape[0]) - 1}

        return cls(
            runner, device, "float32", source, _model=model, _processor=processor, _font_bytes=font_bytes
        )

    def answer(
        self,
        image: Image.Image,
        question: str,
        options: Sequence[str],
        *,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    ) -> dict[str, Any]:
        """Answer one multiple-choice question about one diagram; ``answer`` is the decoded text, stripped."""
        rgb, checked_question, checked_options, checked_tokens = _check_inputs(
            image, question, options, max_new_tokens
        )
        prompt = format_prompt(checked_question, checked_options)
        raw = self._runner(rgb, prompt, checked_tokens)
        if not isinstance(raw, dict) or "answer" not in raw:
            raise RuntimeError("runner must return a dict with 'answer'")
        new_tokens = int(raw.get("new_tokens", 0))
        text = str(raw["answer"]).strip()
        return {
            "answer": text,
            "choice_index": match_option(text, checked_options),
            "question": checked_question,
            "options": checked_options,
            "prompt": prompt,
            "image_size": list(rgb.size),
            "new_tokens": new_tokens,
            "truncated": new_tokens >= checked_tokens,
            "generation": {"max_new_tokens": checked_tokens, "do_sample": False, "decoding": DECODING},
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation contract ---------------------------------------------------------------------------

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._processor is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._processor

    def predict(
        self, records: Sequence[Mapping[str, Any]], *, max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS
    ) -> list[dict[str, Any]]:
        """Answer every validated record's question on its diagram; one `answer` result per record, in order,
        with the record's `id` and `category` attached."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        out = []
        for record in checked:
            with Image.open(record["image"]) as image:
                image.load()
                result = self.answer(
                    image, record["question"], record["options"], max_new_tokens=max_new_tokens
                )
            out.append({"id": record["id"], "category": record["category"], **result})
        return out

    def evaluate(
        self, records: Sequence[Mapping[str, Any]], *, max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS
    ) -> dict[str, Any]:
        """Answer every record and score the matched options against its correct index: accuracy (an
        answer that matches no option counts as wrong), the unmatched rate, the chance level and a
        per-category accuracy."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import mcq_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        results = self.predict(checked, max_new_tokens=max_new_tokens)
        metrics = mcq_metrics(
            [r["choice_index"] for r in results],
            [int(r["answer"]) for r in checked],
            [len(r["options"]) for r in checked],
            categories=[r["category"] for r in checked],
        )
        metrics.update(
            {
                "max_new_tokens": max_new_tokens,
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def _decoder_layers(self) -> int:
        model, _ = self._require_model()
        return int(model.config.text_config.num_layers)

    def _trainable_names(self, trainable_decoder_layers: int) -> list[str]:
        """The last `trainable_decoder_layers` blocks of the answer decoder plus the decoder's final layer
        norm. The untied output projection (`decoder.lm_head`, vocabulary x hidden) and every embedding stay
        frozen, as does the whole image encoder."""
        if (
            isinstance(trainable_decoder_layers, bool)
            or not isinstance(trainable_decoder_layers, int)
            or not 1 <= trainable_decoder_layers <= DECODER_LAYERS
        ):
            raise ValueError(f"trainable_decoder_layers must be an int in 1..{DECODER_LAYERS}")
        model, _ = self._require_model()
        n_layers = self._decoder_layers()
        if trainable_decoder_layers > n_layers:
            raise ValueError(f"trainable_decoder_layers must be an int in 1..{n_layers} for this model")
        first = n_layers - trainable_decoder_layers
        prefixes = tuple(f"decoder.layer.{k}." for k in range(first, n_layers))
        prefixes += ("decoder.final_layer_norm.",)
        return [name for name, _p in model.named_parameters() if name.startswith(prefixes)]

    def _encode_batch(self, records: Sequence[Mapping[str, Any]]) -> tuple[Any, Any]:
        """The frozen encoder's output and patch mask for a batch of (diagram, rendered question) inputs.
        The header is part of the image, so every question is its own encoder input; it is recomputed per
        step under `no_grad` instead of cached (2,048 x 768 floats per question)."""
        import torch

        model, processor = self._require_model()
        device = next(model.parameters()).device
        images = []
        for record in records:
            with Image.open(record["image"]) as image:
                images.append(image.convert("RGB"))
        prompts = [format_prompt(r["question"], r["options"]) for r in records]
        inputs = processor.image_processor(
            images, header_text=prompts, return_tensors="pt", font_bytes=self._font_bytes
        ).to(device)
        with torch.no_grad():
            hidden = model.encoder(
                flattened_patches=inputs["flattened_patches"], attention_mask=inputs["attention_mask"]
            ).last_hidden_state
        return hidden, inputs["attention_mask"]

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 3,
        lr: float = 1e-5,
        batch_size: int = 4,
        trainable_decoder_layers: int = DEFAULT_TRAINABLE_DECODER_LAYERS,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded supervised fine-tuning on validated multiple-choice records.

        Only the last `trainable_decoder_layers` blocks of the answer decoder and the decoder's final layer
        norm train (2 blocks by default); the image encoder, every embedding and the untied output projection
        stay frozen. Each record is one training sample: the question and its numbered options are rendered
        above the diagram exactly as `answer` renders them, the frozen encoder reads the composite, and the
        target is the tokenised text of the correct option with its end-of-sequence token, decoded with
        teacher forcing and scored with the model's own cross-entropy (padding ignored); AdamW at a fixed
        learning rate with gradient clipping at 1.0, no scheduler. Epoch 0 records the frozen model's
        validation metrics; the epoch with the highest validation accuracy is kept (ties keep the earlier)."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 20:
            raise ValueError("epochs must be an int in 1..20")
        if not (0.0 < lr <= 1e-3):
            raise ValueError("lr must be in (0, 1e-3]")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 1..64")
        names = self._trainable_names(trainable_decoder_layers)
        train_checked = validate_dataset(train)["records"]
        val_checked = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"] if val else []
        )
        import torch

        torch.manual_seed(seed)
        model, processor = self._require_model()
        tokenizer = processor.tokenizer
        pad_id = int(tokenizer.pad_token_id)
        started = time.perf_counter()
        wanted = set(names)
        for name, param in model.named_parameters():
            param.requires_grad_(name in wanted)
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
        device = next(model.parameters()).device

        def score_val() -> dict[str, Any] | None:
            if not val_checked:
                return None
            model.eval()
            return {
                k: v
                for k, v in self.evaluate(val_checked).items()
                if k in ("accuracy", "unmatched_rate", "chance", "n")
            }

        history: list[dict[str, Any]] = []
        entry: dict[str, Any] = {"epoch": 0, "train_loss": None, "val": score_val(), "note": "frozen model"}
        history.append(entry)
        if progress:
            progress(entry)
        best_score = entry["val"]["accuracy"] if entry["val"] else -math.inf
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
        initial_state = {k: v.clone() for k, v in best_state.items()}
        best_epoch = 0
        generator = torch.Generator().manual_seed(seed)
        try:
            for epoch in range(1, epochs + 1):
                model.train()
                order = torch.randperm(len(train_checked), generator=generator).tolist()
                losses = []
                for start in range(0, len(order), batch_size):
                    chosen = [train_checked[j] for j in order[start : start + batch_size]]
                    hidden, mask = self._encode_batch(chosen)
                    targets = tokenizer(
                        [r["options"][int(r["answer"])] for r in chosen],
                        padding=True,
                        truncation=True,
                        max_length=MAX_NEW_TOKENS + 1,
                        return_tensors="pt",
                    ).to(device)
                    labels = targets["input_ids"].masked_fill(targets["input_ids"] == pad_id, -100)
                    out = model(encoder_outputs=(hidden,), attention_mask=mask, labels=labels)
                    optimiser.zero_grad(set_to_none=True)
                    out.loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    optimiser.step()
                    losses.append(float(out.loss.detach()))
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": score_val()}
                history.append(entry)
                if progress:
                    progress(entry)
                current = entry["val"]["accuracy"] if entry["val"] else math.inf
                if current > best_score or not entry["val"]:
                    best_score = current
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in wanted}
                    best_epoch = epoch
        except BaseException:
            # Transactional: a failure in training, validation or the progress callback leaves the base
            # exactly as it was, with every parameter frozen again.
            restore = dict(model.state_dict())
            restore.update(initial_state)
            model.load_state_dict(restore, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable_decoder_layers": trainable_decoder_layers,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": (
                "highest validation accuracy" if val_checked else "final epoch (no validation split)"
            ),
            "lr": lr,
            "batch_size": batch_size,
            "n_train": len(train_checked),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted answer-decoder blocks and final norm as safetensors plus a base manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHT_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def _check_artifact_manifest(self, root: Path, manifest: Mapping[str, Any]) -> Path:
        """Refuse an artifact whose manifest is not exactly the one this pipeline writes: the supported format
        and version, the pinned base (id, revision, weight file, digest), exactly one file entry named
        `adapter.safetensors` that resolves inside the artifact directory, and a recorded
        `trainable_decoder_layers` in range. Nothing is deserialised here. The digest check that follows
        detects corruption or drift of the weights relative to the adjacent manifest; it is not authenticity
        against an actor who can replace both files."""
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(
                f"artifact format_version {manifest.get('format_version')!r} is not the supported "
                f"{ARTIFACT_FORMAT_VERSION!r}"
            )
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        if base.get("weight_file", WEIGHT_FILE) != WEIGHT_FILE:
            raise ValueError("artifact was adapted from a different base weight file")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact manifest must name exactly {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weight path must resolve inside the artifact directory")
        adapter = manifest.get("adapter")
        layers = adapter.get("trainable_decoder_layers") if isinstance(adapter, Mapping) else None
        if isinstance(layers, bool) or not isinstance(layers, int) or not 1 <= layers <= DECODER_LAYERS:
            raise ValueError("artifact manifest does not record an in-range integer trainable_decoder_layers")
        if not isinstance(manifest.get("tensors"), list):
            raise ValueError("artifact manifest must list its tensors")
        return weights_path

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, digest and exact tensor set **before** deserialising, then overwrite
        exactly the tensors it carries."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path = self._check_artifact_manifest(root, manifest)
        entry = manifest["files"][0]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        # The exact tensor set the recorded configuration implies — no subset, no extra, no other layer.
        expected = sorted(self._trainable_names(manifest["adapter"]["trainable_decoder_layers"]))
        if sorted(manifest["tensors"]) != expected:
            raise ValueError("artifact tensor list does not match its recorded configuration")
        model, _ = self._require_model()
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from its manifest")
        state = model.state_dict()
        for key, value in tensors.items():
            if key not in state or not key.startswith("decoder."):
                raise ValueError(
                    f"artifact tensor {key} is not an adaptable answer-decoder tensor of the base"
                )
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key} has shape {tuple(value.shape)}, "
                    f"base has {tuple(state[key].shape)}"
                )
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        model.load_state_dict(merged, strict=True)
        model.eval()
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": manifest["tensors"],
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Pix2StructAI2DPipeline:
        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 3/3:** `src/pix2struct_ai2d_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Multiple-choice diagram dataset contract for fine-tuning: the pinned AI2D sample, validation, seeded
image-disjoint splitting, BYOD loaders and JSONL export.

The default dataset is **real** and from the checkpoint's own domain: questions from the AI2D test split
(Kembhavi et al., ECCV 2016 — school-science diagrams with multiple-choice questions) as mirrored on the
Hugging Face Hub in `lmms-lab-encoder/ai2d`. `google/pix2struct-ai2d-base` was fine-tuned on AI2D's
*training* questions, so this is continued adaptation inside the domain on questions it was not trained on,
not a distribution shift. The sample is one pinned parquet shard (`data/test-00000-of-00002.parquet`,
62,292,686 bytes) downloaded whole at the pinned dataset revision and refused unless its SHA-256 matches the
pin before `pyarrow` reads a byte of it; the diagrams are written to the cache under their own content
digest. A seeded subset of whole diagrams is drawn from it and cut **by diagram** into training,
validation and test questions, so no test diagram is ever trained on.

A record is ``{id, image_id, image, question, options, answer, category}`` — the path of the diagram, the
question, 2..6 distinct options, the zero-based index of the correct option, and a category (`letter-label`
when every option is a one- or two-character diagram label such as `A` or `d`, `text-option` otherwise;
BYOD records may carry any label, `other` by default). Several questions share one diagram; records on the
same `image_id` are always kept in one split.

The shard's SHA-256 and the counts it yields are recorded by `tools/pin_corpus.py` (it needs Hub access);
until they are recorded, `fetch_corpus` refuses to read the shard rather than read an unpinned file.
"""

from __future__ import annotations

import hashlib
import io
import json
import random
import re
from collections import Counter
from collections.abc import Callable, Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image

# standalone rewrite (build_notebook.py): `from .pipeline import (` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "AI2D (test questions)"
CORPUS_REPO = "lmms-lab-encoder/ai2d"
CORPUS_REVISION = "c83a9b9692933aff8349157c88a413df9d02c4e5"
CORPUS_RELEASE = (
    "AI2D test split (3,088 questions) as mirrored on the Hugging Face Hub, dataset revision c83a9b96"
)
CORPUS_LICENSE = (
    "not declared by the Hub mirror; AI2D is published by the Allen Institute for AI "
    "(Kembhavi et al. 2016) — check its terms before redistributing the diagrams or an adapter "
    "trained on them"
)
CORPUS_COLUMNS = ("question", "options", "answer", "image")
# The shard's pins. `sha256`, `rows` and `images` are written by tools/pin_corpus.py from a verified download;
# while `sha256` is None the reader refuses to run.
CORPUS_FILE: dict[str, Any] = {
    "path": "data/test-00000-of-00002.parquet",
    "bytes": 62_292_686,
    "sha256": "450ecfa95b0c475ba214cd9a33b7ec5d1d782e7321a54fc652453d8776743702",
    "rows": 1544,
    "images": 391,
}
DEFAULT_CACHE_DIR = Path("weights") / "ai2d"
SAMPLE_SEED = 42
# Target question counts per split; whole diagrams are allocated until each target is reached, so the
# realised counts can exceed a target by the questions of the last diagram added.
SAMPLE_QUESTIONS = {"train": 360, "validation": 80, "test": 160}
MIN_RECORDS = 8
MAX_RECORDS = 5_000
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")
_LABEL_MAX_CHARS = 2


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def corpus_pinned() -> bool:
    """Whether the shard's SHA-256 has been recorded (see tools/pin_corpus.py)."""
    return isinstance(CORPUS_FILE.get("sha256"), str) and len(CORPUS_FILE["sha256"]) == 64


def _hub_download(cache: Path) -> Path:
    from huggingface_hub import hf_hub_download

    return Path(
        hf_hub_download(
            CORPUS_REPO,
            CORPUS_FILE["path"],
            repo_type="dataset",
            revision=CORPUS_REVISION,
            local_dir=str(cache),
        )
    )


def fetch_corpus(
    *, cache_dir: str | Path | None = None, downloader: Callable[[Path], Path] | None = None
) -> Path:
    """Return the path of the pinned shard, downloading it at the pinned revision when the cached copy is
    absent or drifted; refused on any size or SHA-256 mismatch, and outright while no pin is recorded."""
    if not corpus_pinned():
        raise RuntimeError(
            f"{CORPUS_REPO}@{CORPUS_REVISION[:8]} {CORPUS_FILE['path']}: no SHA-256 pin is recorded; run "
            "tools/pin_corpus.py with Hub access to record it before the sample can be read"
        )
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    local = cache / CORPUS_FILE["path"]

    def ok(path: Path) -> bool:
        return (
            path.is_file()
            and path.stat().st_size == CORPUS_FILE["bytes"]
            and _sha256_file(path) == CORPUS_FILE["sha256"]
        )

    if ok(local):
        return local
    fetched = (downloader or _hub_download)(cache)
    if not ok(fetched):
        size = fetched.stat().st_size if fetched.is_file() else None
        raise ValueError(
            f"{CORPUS_FILE['path']}: fetched {size} bytes, pinned {CORPUS_FILE['bytes']} / "
            f"{CORPUS_FILE['sha256'][:16]}…; refusing to read it"
        )
    return fetched


def read_corpus(path: str | Path) -> list[dict[str, Any]]:
    """The shard's rows as ``{question, options, answer, image_bytes}`` (answer as a zero-based int)."""
    import pyarrow.parquet as pq

    rows = pq.read_table(str(path), columns=list(CORPUS_COLUMNS)).to_pylist()
    out = []
    for index, row in enumerate(rows):
        image = row["image"]
        data = image.get("bytes") if isinstance(image, Mapping) else None
        if not data:
            raise ValueError(f"row {index}: no image bytes")
        out.append(
            {
                "question": str(row["question"]),
                "options": [str(o) for o in row["options"]],
                "answer": int(str(row["answer"]).strip()),
                "image_bytes": bytes(data),
            }
        )
    if CORPUS_FILE.get("rows") is not None and len(out) != CORPUS_FILE["rows"]:
        raise ValueError(f"shard has {len(out)} rows, pinned {CORPUS_FILE['rows']}")
    return out


def option_category(options: Sequence[str]) -> str:
    """`letter-label` when every option is a short diagram label (`A`, `d`, `12`), else `text-option`."""
    return "letter-label" if all(len(str(o).strip()) <= _LABEL_MAX_CHARS for o in options) else "text-option"


def _image_suffix(data: bytes) -> str:
    with Image.open(io.BytesIO(data)) as image:
        fmt = (image.format or "PNG").lower()
    return {"jpeg": ".jpg", "png": ".png", "gif": ".gif", "webp": ".webp"}.get(fmt, ".png")


def build_sample_dataset(
    rows: Sequence[Mapping[str, Any]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
    image_dir: str | Path | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Group the rows by diagram (content digest), shuffle the diagrams with `seed`, and allocate whole
    diagrams to test, validation and train until each split's question target is reached. Questions that
    `validate_dataset` would reject (an option over `MAX_OPTION_CHARS`, options that coincide after
    normalisation, …) are left out before they are counted, so every split validates. The diagrams of the
    chosen questions are written to `image_dir` under their digest."""
    sizes = dict(sizes or SAMPLE_QUESTIONS)
    out_dir = Path(image_dir) if image_dir is not None else DEFAULT_CACHE_DIR / "images"
    out_dir.mkdir(parents=True, exist_ok=True)
    groups: dict[str, list[Mapping[str, Any]]] = {}
    for row in rows:
        groups.setdefault(_sha256_bytes(row["image_bytes"])[:16], []).append(row)
    if CORPUS_FILE.get("images") is not None and len(groups) != CORPUS_FILE["images"]:
        raise ValueError(f"shard has {len(groups)} distinct diagrams, pinned {CORPUS_FILE['images']}")
    order = sorted(groups)
    random.Random(seed).shuffle(order)
    out: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    for image_id in order:
        open_splits = [name for name in ("test", "validation", "train") if len(out[name]) < sizes[name]]
        target = open_splits[0] if open_splits else None
        if target is None:
            break
        data = groups[image_id][0]["image_bytes"]
        path = out_dir / f"{image_id}{_image_suffix(data)}"
        if not path.is_file() or _sha256_bytes(path.read_bytes())[:16] != image_id:
            path.write_bytes(data)
        for row in groups[image_id]:
            record = {
                "id": f"{target}-{len(out[target]):04d}",
                "image_id": image_id,
                "image": str(path),
                "question": str(row["question"]),
                "options": [str(o) for o in row["options"]],
                "answer": int(row["answer"]),
                "category": option_category(row["options"]),
            }
            try:
                _check_record(record, 0, base_dir=None)
            except ValueError:
                continue  # outside the ceilings `validate_dataset` and `answer` apply: not part of the sample
            out[target].append(record)
    short = {name: (len(out[name]), sizes[name]) for name in out if len(out[name]) < sizes[name]}
    if short:
        raise ValueError(f"the shard's diagrams do not fill the split targets: {short}")
    return {"train": out["train"], "validation": out["validation"], "test": out["test"]}


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    downloader: Callable[[Path], Path] | None = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned shard."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    rows = read_corpus(fetch_corpus(cache_dir=cache, downloader=downloader))
    return build_sample_dataset(rows, seed=seed, sizes=sizes, image_dir=cache / "images")


def _check_record(record: Any, index: int, *, base_dir: Path | None) -> dict[str, Any]:
    label = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label} must be a mapping with id/image/question/options/answer")
    for key in ("id", "image", "question", "options", "answer"):
        if key not in record:
            raise ValueError(f"{label} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label}: id must match {_ID_RE.pattern}")
    image_ref = record["image"]
    if not isinstance(image_ref, (str, Path)) or not str(image_ref).strip():
        raise ValueError(f"{label}: image must be a file path")
    path = Path(image_ref)
    if not path.is_absolute() and base_dir is not None:
        path = base_dir / path
    if not path.is_file():
        raise ValueError(f"{label}: image file not found: {path}")
    try:
        with Image.open(path) as handle:
            handle.load()
            validate_image(handle)
            width, height = handle.size
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{label}: {exc}") from exc
    except OSError as exc:
        raise ValueError(f"{label}: image cannot be decoded: {exc}") from exc
    question = record["question"]
    if not isinstance(question, str) or not " ".join(question.split()):
        raise ValueError(f"{label}: question must be a non-empty string")
    question = " ".join(question.split())
    if len(question) > MAX_QUESTION_CHARS:
        raise ValueError(f"{label}: question exceeds MAX_QUESTION_CHARS={MAX_QUESTION_CHARS}")
    options = record["options"]
    if isinstance(options, str) or not isinstance(options, Sequence):
        raise ValueError(f"{label}: options must be a list of strings")
    if not MIN_OPTIONS <= len(options) <= MAX_OPTIONS:
        raise ValueError(f"{label}: options must have {MIN_OPTIONS}..{MAX_OPTIONS} entries")
    checked_options = [" ".join(str(o).split()) if isinstance(o, str) else "" for o in options]
    if not all(checked_options):
        raise ValueError(f"{label}: every option must be a non-empty string")
    if any(len(o) > MAX_OPTION_CHARS for o in checked_options):
        raise ValueError(f"{label}: an option exceeds MAX_OPTION_CHARS={MAX_OPTION_CHARS}")
    if len({normalize_answer(o) for o in checked_options}) != len(checked_options):
        raise ValueError(f"{label}: options must be distinct after normalisation")
    answer = record["answer"]
    if isinstance(answer, bool) or not isinstance(answer, int) or not 0 <= answer < len(checked_options):
        raise ValueError(f"{label}: answer must be the zero-based index of the correct option")
    return {
        "id": rid,
        "image_id": str(record.get("image_id", path.name)),
        "image": str(path),
        "image_size": [width, height],
        "question": question,
        "options": checked_options,
        "answer": answer,
        "category": str(record.get("category", "other")),
    }


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
    base_dir: str | Path | None = None,
) -> dict[str, Any]:
    """Structural validation of a multiple-choice dataset (every diagram opened and decoded, every question
    held to the same ceilings `answer` applies); raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError("records must be a list of {id, image, question, options, answer} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    base = Path(base_dir) if base_dir is not None else None
    checked = []
    ids: set[str] = set()
    for index, record in enumerate(records):
        item = _check_record(record, index, base_dir=base)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        checked.append(item)
    return {
        "records": checked,
        "n_records": len(checked),
        "unique_images": len({r["image_id"] for r in checked}),
        "categories": dict(Counter(r["category"] for r in checked)),
        "options_per_question": {
            "min": min(len(r["options"]) for r in checked),
            "max": max(len(r["options"]) for r in checked),
        },
        "answer_positions": dict(sorted(Counter(r["answer"] for r in checked).items())),
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [
        [r["id"], r["image_id"], r["question"], list(r["options"]), int(r["answer"]), r.get("category", "")]
        for r in records
    ]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no diagram appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = str(record.get("image_id", record["id"]))
            if key in seen and seen[key] != name:
                raise ValueError(f"diagram {key!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
    base_dir: str | Path | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded split of a BYOD dataset into train/validation/test **by diagram**: every question on the same
    diagram lands in the same split, so a test diagram is never seen in training."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records, base_dir=base_dir)["records"]
    groups: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        groups.setdefault(record["image_id"], []).append(record)
    order = list(groups.values())
    random.Random(seed).shuffle(order)
    n_test = max(1, round(len(checked) * test_fraction))
    n_val = round(len(checked) * val_fraction)
    splits: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    for group in order:
        if len(splits["test"]) < n_test:
            splits["test"].extend(group)
        elif len(splits["validation"]) < n_val:
            splits["validation"].extend(group)
        else:
            splits["train"].extend(group)
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read records from a JSON array or a JSONL file of ``{id, image, question, options, answer}`` objects;
    `image` paths are resolved relative to the file's directory by `validate_dataset(..., base_dir=...)`."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"dataset not found: {file_path}")
    suffix = file_path.suffix.lower()
    text = file_path.read_text(encoding="utf-8")
    if suffix == ".jsonl":
        return [json.loads(line) for line in text.splitlines() if line.strip()]
    if suffix == ".json":
        data = json.loads(text)
        if not isinstance(data, list):
            raise ValueError("JSON dataset must be an array of records")
        return data
    raise ValueError("BYOD datasets must be .json or .jsonl")


def write_dataset_jsonl(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """One record per line in the shape `load_byod_dataset` reads back (image paths as given)."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    keys = ("id", "image_id", "image", "question", "options", "answer", "category")
    with open(out, "w", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps({k: record[k] for k in keys if k in record}, ensure_ascii=False) + "\n")
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `0d6b2606efe0…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Pix2StructAI2DPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "pix2struct-ai2d-base",
  "modelId": "google/pix2struct-ai2d-base",
  "revision": "0d6b2606efe05c77c1d0670647740802a9e68eef",
  "files": [
    {
      "path": "README.md",
      "bytes": 7085,
      "sha256": "1af9a1edb7519ba201f476cf255802a3fa163e081e3014140f0293e1ab701d5f"
    },
    {
      "path": "config.json",
      "bytes": 4889,
      "sha256": "bbb61790293483693c82c55f4c41c8c79514297dae72c2454d85b8cdf2e21d2d"
    },
    {
      "path": "model.safetensors",
      "bytes": 564606744,
      "sha256": "652c92f8b995d9fc5ea46350c854fd88767bf36c391552f126f6e7487b0f27a3"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 249,
      "sha256": "9662520ca2d4e38fccb8465ea3b443a88ced7079e6cb2b6e0230bae98c83b5c1"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 2201,
      "sha256": "5c87151ef0f72a99d1f766a4c418bd2a1f90aaa30a8e22fe5eca9641daebb64f"
    },
    {
      "path": "spiece.model",
      "bytes": 851388,
      "sha256": "7fd650335add59bed55a432186ca0437a09e185c2d241faab468a538fe6bcf94"
    },
    {
      "path": "tokenizer.json",
      "bytes": 3265159,
      "sha256": "0af109b23840545ef2c286073f4373959badba1faa73c8557881d5126f6287c9"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 2583,
      "sha256": "5fdb6767a49aca48fdfa43d0279321918185fc4997bdb3ea72bf3a6301a1b43d"
    }
  ],
  "totalBytes": 568740298
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = Pix2StructAI2DPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. AI2D questions, diagrams and split

`fetch_corpus` returns the pinned shard from the cache under `weights/ai2d/` or downloads it at the pinned dataset revision, and refuses it unless its byte size and SHA-256 equal the pins in the carried module (it also refuses to run at all while no SHA-256 pin is recorded). `read_corpus` reads the question, options, answer and image columns with `pyarrow`. `build_sample_dataset` groups the questions by diagram (the SHA-256 of the image bytes), shuffles the diagrams with `SPLIT_SEED` and allocates **whole diagrams** to the test, validation and training splits until each reaches its question target (`SAMPLE_QUESTIONS`), leaving out the few questions `validate_dataset` would reject (an option longer than `MAX_OPTION_CHARS`, or options that coincide after normalisation), labelling each question `letter-label` when every option is a one- or two-character diagram label and `text-option` otherwise. `validate_dataset` then opens and decodes every diagram and checks every question against the contract, `check_split_disjoint` asserts no diagram is shared, and the training split is written to `outputs/pix2struct_ai2d_train.jsonl` in the shape BYOD expects.

Look for: the shard's row count, the question and diagram counts per split, the category mix, the distribution of correct positions (the position-prior baseline in Section 6 is built from it), three digests, and four refusal probes — a duplicate id, a missing image file, an answer index out of range and a dataset too small to split — each rejected before `torch` does anything.

In [ ]:
import collections
import hashlib
import io
import json
import zipfile

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_dir = Path('work') / 'byod'
    byod_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        for member in archive.infolist():
            name = Path(member.filename).name
            if member.is_dir() or not name or name.startswith('.'):
                continue
            (byod_dir / name).write_bytes(archive.read(member))
    records_file = next(p for p in (byod_dir / 'records.jsonl', byod_dir / 'records.json') if p.is_file())
    records = load_byod_dataset(records_file)
    splits = split_dataset(records, seed=SPLIT_SEED, base_dir=byod_dir)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records)}
else:
    shard_path = fetch_corpus(cache_dir='weights/ai2d')
    corpus_rows = read_corpus(shard_path)
    raw_rows = {'questions': len(corpus_rows), 'diagrams': len({hashlib.sha256(r['image_bytes']).hexdigest() for r in corpus_rows})}
    splits = build_sample_dataset(corpus_rows, seed=SPLIT_SEED, image_dir='weights/ai2d/images')
    data_source = f'{CORPUS_NAME} — {CORPUS_RELEASE}'
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint(splits)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
categories = {name: manifest['categories'] for name, manifest in dataset_manifests.items()}
write_dataset_jsonl(splits['train'], 'outputs/pix2struct_ai2d_train.jsonl')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint, 'shard_sha256': str(CORPUS_FILE['sha256'])[:16] + '...'})
for name, manifest in dataset_manifests.items():
    print({name: {'questions': manifest['n_records'], 'diagrams': manifest['unique_images'], 'categories': manifest['categories'], 'answer_positions': manifest['answer_positions'], 'options_per_question': manifest['options_per_question'], 'digest': manifest['digest'][:16] + '...'}})
example = splits['train'][0]
print({'example': {'id': example['id'], 'image': Path(example['image']).name, 'size': example['image_size'], 'category': example['category'], 'question': example['question'], 'options': example['options'], 'answer': example['answer']}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in splits['train'][:8]],
    'missing image file': [{**splits['train'][0], 'image': 'work/does-not-exist.png'}, *splits['train'][1:8]],
    'answer out of range': [{**splits['train'][0], 'answer': len(splits['train'][0]['options'])}, *splits['train'][1:8]],
    'too small': splits['train'][:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Answer through the inference contract

The inference contract is exercised as the inference-only tutorial exercised it: a flat cartoon plant diagram drawn in code at 640×640 — a pink flower, two leaves, a stem, four roots in brown soil and a sun — with six numbered markers joined to their parts (the AI2D convention: numbers on the diagram, words only in the options), and ten authored questions with their correct options. It is a different image family from the AI2D diagrams, and the adapted model will be asked the same questions in Section 9. `validate_inputs` applies exactly the checks `answer` applies (image sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE`, a non-empty question of at most `MAX_QUESTION_CHARS`, `MIN_OPTIONS`..`MAX_OPTIONS` distinct options of at most `MAX_OPTION_CHARS`, `max_new_tokens` in `[1, MAX_NEW_TOKENS]`) and returns an input manifest; a question with a single option is validated too and its rejection recorded as a finding. `answer` returns the decoded text, the matched `choice_index` (or `null` when the text equals no option — there is no fuzzy match), the rendered prompt, `new_tokens`, a `truncated` flag and the model identity. **No score exists.** As recorded in the model card, the inference-only smoke answered four of the ten correctly (chance is one in four) and said `root` to almost every label question whenever `root` was an option. `evaluation_report` scores those ten against the options you drew yourself — verdict `sample-sanity`, plumbing evidence, not a metric; whether answers are *right* on real diagrams is what Section 6 measures on the test questions. The image digest depends on the Pillow build's bundled font rendering.

In [ ]:
import math
import time

import numpy as np
from PIL import Image, ImageDraw, ImageFont

ANSWER_MAX_TOKENS = 16  # @param {type:"integer"}


def synthetic_diagram(size=640):
    """A cartoon plant diagram with six numbered markers; returns image + [(question, options, correct index)]."""
    image = Image.new('RGB', (size, size), 'white')
    d = ImageDraw.Draw(image)
    marker_font = ImageFont.load_default(size=34)
    d.rectangle([0, 440, 640, 640], fill=(160, 120, 70))  # soil
    d.ellipse([500, 40, 600, 140], fill=(255, 215, 0))  # sun
    d.rectangle([310, 220, 330, 440], fill=(40, 140, 40))  # stem
    d.polygon([(310, 330), (220, 290), (240, 350)], fill=(50, 170, 50))  # left leaf
    d.polygon([(330, 380), (420, 340), (400, 400)], fill=(50, 170, 50))  # right leaf
    for k in range(6):  # petals
        a = math.radians(60 * k)
        cx, cy = 320 + 45 * math.cos(a), 190 + 45 * math.sin(a)
        d.ellipse([cx - 22, cy - 22, cx + 22, cy + 22], fill=(230, 60, 120))
    d.ellipse([298, 168, 342, 212], fill=(255, 200, 40))  # flower centre
    for dx in (-60, -20, 25, 70):  # roots
        d.line([(320, 440), (320 + dx, 560)], fill=(120, 80, 40), width=5)
    markers = [('1', (140, 150), (275, 175)), ('2', (90, 330), (225, 320)), ('3', (470, 250), (332, 300)), ('4', (140, 560), (290, 520)), ('5', (550, 175), (550, 140)), ('6', (560, 600), (560, 600))]
    for text, pos, tip in markers:
        if pos != tip:
            d.line([pos, tip], fill='black', width=3)
        d.rectangle([pos[0] - 22, pos[1] - 22, pos[0] + 22, pos[1] + 22], fill='white', outline='black', width=2)
        d.text(pos, text, fill='black', font=marker_font, anchor='mm')
    qa = [
        ('What does the label 1 represent?', ['flower', 'leaf', 'stem', 'root'], 0),
        ('What does the label 2 represent?', ['flower', 'leaf', 'stem', 'root'], 1),
        ('What does the label 3 represent?', ['root', 'stem', 'leaf', 'flower'], 1),
        ('What does the label 4 represent?', ['stem', 'flower', 'root', 'leaf'], 2),
        ('What does the label 5 represent?', ['moon', 'sun', 'cloud', 'rain'], 1),
        ('What does the label 6 represent?', ['water', 'air', 'soil', 'rock'], 2),
        ('Which part of the plant is below the soil?', ['flower', 'leaf', 'stem', 'root'], 3),
        ('What provides light to the plant?', ['soil', 'sun', 'root', 'leaf'], 1),
        ('Which part connects the roots to the flower?', ['leaf', 'soil', 'stem', 'sun'], 2),
        ('Which part is at the top of the plant?', ['root', 'stem', 'flower', 'soil'], 2),
    ]
    return image, qa


diagram, qa = synthetic_diagram()
diagram_name = 'synthetic_plant_diagram_640x640.png'
diagram_questions = [{'question': q, 'options': options} for q, options, _ in qa]
diagram_correct = [gold for _, _, gold in qa]
diagram_sha256 = hashlib.sha256(np.asarray(diagram.convert('RGB')).tobytes()).hexdigest()
ceilings = {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_PATCHES': MAX_PATCHES, 'MAX_QUESTION_CHARS': MAX_QUESTION_CHARS, 'MIN_OPTIONS': MIN_OPTIONS, 'MAX_OPTIONS': MAX_OPTIONS, 'MAX_OPTION_CHARS': MAX_OPTION_CHARS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'DECODING': DECODING, 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS}
print(ceilings)
input_manifest = validate_inputs(diagram, diagram_questions, max_new_tokens=ANSWER_MAX_TOKENS, names=[diagram_name])
try:
    validate_inputs(diagram, [{'question': 'What is this?', 'options': ['a plant']}])
except ValueError as exc:
    input_manifest['findings'].append({'input': 'single-option-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/pix2struct_ai2d_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'diagram': diagram_name, 'rgb_sha256': diagram_sha256[:16] + '...', 'questions': len(diagram_questions), 'manifest_verdict': input_manifest['verdict'], 'findings': len(input_manifest['findings'])})
results = []
for entry in diagram_questions:
    started = time.perf_counter()
    result = pipe.answer(diagram, entry['question'], entry['options'], max_new_tokens=ANSWER_MAX_TOKENS)
    results.append({'seconds': round(time.perf_counter() - started, 3), **result})
for result in results:
    matched = result['options'][result['choice_index']] if result['choice_index'] is not None else 'NO OPTION MATCHED'
    print(f"Q: {result['question']}  options={result['options']}\n   A: {result['answer']!r} -> {matched}  ({result['new_tokens']} tokens{', TRUNCATED' if result['truncated'] else ''})")
checks = {
    'one_result_per_question': len(results) == len(diagram_questions),
    'answers_are_text': all(isinstance(r['answer'], str) for r in results),
    'budget_respected': all(r['new_tokens'] <= ANSWER_MAX_TOKENS for r in results),
    'setting_echoed': all(r['generation']['max_new_tokens'] == ANSWER_MAX_TOKENS and r['generation']['do_sample'] is False for r in results),
}
if not all(checks.values()):
    raise RuntimeError(f'answer output failed a sanity check: {checks}')
frozen_diagram = evaluation_report(results, diagram_correct, sample_kind='synthetic')
print({'checks': checks, 'frozen_diagram_verdict': frozen_diagram['verdict'], 'accuracy': frozen_diagram['metrics'][0]['value'], 'chance': frozen_diagram['baselines'][0]['value'], 'unmatched': sum(r['choice_index'] is None for r in results)})

## 6. Baselines and the frozen model's accuracy on the test questions

Four references frame the adaptation. **Chance** is the mean of 1/options over the test questions — what uniform guessing scores. The **position-prior baseline** answers every test question with the option position most often correct in the training split (printed in Section 4). The **longest-option baseline** answers with the longest option text. Neither baseline looks at the diagram, so a model that does not beat them has not shown that it reads the diagram. The **frozen model** answers every test question with the budget from Section 5 and is scored by `pipe.evaluate`: **accuracy** (the matched option is the correct one; an answer that matches no option counts as wrong), the **unmatched rate** and a per-category accuracy (`letter-label` questions name lettered parts of the diagram; `text-option` questions have words as options). The checkpoint was fine-tuned on AI2D's training questions, so expect it well above chance here; whether it is, is recorded as `frozen_beats_chance` rather than assumed. The measured values of the first clean run are recorded in `docs/release-verification.md` and the model card.

In [ ]:
baseline_position = position_prior_baseline(train_records, test_records)
baseline_longest = longest_option_baseline(test_records)
print({'position_prior_baseline': round(baseline_position['accuracy'], 3), 'note': baseline_position['note'], 'n': baseline_position['n']})
print({'longest_option_baseline': round(baseline_longest['accuracy'], 3), 'chance': round(baseline_longest['chance'], 3)})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records, max_new_tokens=ANSWER_MAX_TOKENS)
print({'frozen_model_test': {'accuracy': round(frozen_test['accuracy'], 3), 'unmatched_rate': round(frozen_test['unmatched_rate'], 3), 'chance': round(frozen_test['chance'], 3)}, 'n': frozen_test['n'], 'verdict': frozen_test['verdict'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'by_category': {'position_prior': baseline_position['by_category'], 'frozen': frozen_test['by_category']}})
print({'definitions': frozen_test['definitions']})
frozen_predictions = {r['id']: r for r in pipe.predict(test_records[:3], max_new_tokens=ANSWER_MAX_TOKENS)}
for record in test_records[:3]:
    print({'category': record['category'], 'question': record['question'], 'options': record['options'], 'correct': record['options'][record['answer']], 'frozen': frozen_predictions[record['id']]['answer']})
frozen_beats_chance = frozen_test['accuracy'] > frozen_test['chance']
print({'frozen_beats_chance': frozen_beats_chance})

## 7. Bounded fine-tuning of the answer decoder's last blocks

`pipe.adapt` trains only the last `TRAINABLE_DECODER_LAYERS` blocks of the answer decoder plus the decoder's final layer norm — two blocks by default, 18,879,744 of 282,285,696 parameters; the image encoder, every embedding and the untied output projection (a 50,244 × 768 matrix) stay frozen. Each training question is one sample: its question and numbered options are rendered above its diagram exactly as `answer` renders them, the frozen encoder reads the composite (recomputed each step without gradients — the header makes every question its own image), and the target is the tokenised text of the correct option with its end-of-sequence token, decoded with teacher forcing and scored with the model's own cross-entropy (padding ignored); AdamW at a fixed learning rate, gradient clipping at 1.0, seeded shuffling and no scheduler. Epoch 0 records the frozen model's validation accuracy; every epoch is scored on the validation questions and the epoch with the highest validation accuracy is kept (ties keep the earlier one). A validation split of about eighty questions makes that selection coarse — one question is more than a point of accuracy — which is why the held-out split in Section 8 is what the numbers are read from. If no epoch beats the frozen model on validation, the selector keeps epoch 0 and the adapter reproduces the frozen answers; that outcome is reported, not hidden.

In [ ]:
EPOCHS = 3  # @param {type:"integer"}
LEARNING_RATE = 1e-5  # @param {type:"number"}
BATCH_SIZE = 4  # @param {type:"integer"}
TRAINABLE_DECODER_LAYERS = 2  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row.update({'val_accuracy': round(entry['val']['accuracy'], 3), 'val_unmatched_rate': round(entry['val']['unmatched_rate'], 3)})
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_decoder_layers=TRAINABLE_DECODER_LAYERS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'training_questions': adapt_result['n_train'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test questions were never used for training or epoch selection, and no test diagram appears in the training or validation splits. The adapted model is scored exactly as the frozen model was in Section 6, and the five systems — chance, the two baselines, the frozen and the adapted model — are put side by side overall and per category. Read it in this order: **accuracy** first (the metric the epoch was selected on), then the **unmatched rate** (an adaptation that teaches the decoder to copy option text exactly lowers it, which raises accuracy without the model reading the diagram any better), then the `letter-label` / `text-option` split. `adapted_beats_frozen` records whether held-out accuracy rose. A test split of about 160 questions from one seeded draw of one shard gives **no dispersion estimate** — one question is more than half a point — so the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a gain on AI2D says nothing about your diagrams until you measure it there.

In [ ]:
adapted_test = pipe.evaluate(test_records, max_new_tokens=ANSWER_MAX_TOKENS)
adapted_val = pipe.evaluate(val_records, max_new_tokens=ANSWER_MAX_TOKENS)
comparison = {
    'accuracy': {'chance': round(frozen_test['chance'], 3), 'position_prior': round(baseline_position['accuracy'], 3), 'longest_option': round(baseline_longest['accuracy'], 3), 'frozen': round(frozen_test['accuracy'], 3), 'adapted': round(adapted_test['accuracy'], 3)},
    'unmatched_rate': {'frozen': round(frozen_test['unmatched_rate'], 3), 'adapted': round(adapted_test['unmatched_rate'], 3)},
    'delta_vs_frozen': {'accuracy': round(adapted_test['accuracy'] - frozen_test['accuracy'], 3), 'unmatched_rate': round(adapted_test['unmatched_rate'] - frozen_test['unmatched_rate'], 3)},
    'by_category': {category: {'n': row['n'], 'position_prior': round(baseline_position['by_category'][category]['accuracy'], 3), 'frozen': round(row['accuracy'], 3), 'adapted': round(adapted_test['by_category'][category]['accuracy'], 3)} for category, row in frozen_test['by_category'].items()},
}
for key, row in comparison.items():
    print({key: row})
adapted_predictions = {r['id']: r for r in pipe.predict(test_records[:3], max_new_tokens=ANSWER_MAX_TOKENS)}
for record in test_records[:3]:
    print({'question': record['question'], 'correct': record['options'][record['answer']], 'frozen': frozen_predictions[record['id']]['answer'], 'adapted': adapted_predictions[record['id']]['answer']})
adapted_beats_frozen = adapted_test['accuracy'] > frozen_test['accuracy']
print({'adapted_beats_frozen': adapted_beats_frozen})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'categories': categories,
    'max_new_tokens': ANSWER_MAX_TOKENS,
    'baselines': {'position_prior': baseline_position, 'longest_option': baseline_longest},
    'frozen_test': frozen_test,
    'frozen_beats_chance': frozen_beats_chance,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
    'adapted_beats_frozen': adapted_beats_frozen,
}
with open('outputs/pix2struct_ai2d_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
print({'report': 'outputs/pix2struct_ai2d_evaluation_report.json'})

## 9. Re-answer the drawn diagram, export the adapter and reload it

The ten questions on the drawn plant from Section 5 are answered again by the adapted model and scored against the options you drew — a different image family from the AI2D diagrams it was tuned on, so this is a small look at whether the adaptation changed the model's behaviour *outside* its sample (ten questions of evidence, not a measurement; a different answer here is a finding to record, not a failure). Both answer sets are written as CSV.

`pipe.save_artifact` writes the trained tensors — the answer decoder's last two blocks and its final layer norm, about 76 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `Pix2StructAI2DPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, refuses any tensor that is not an answer-decoder tensor, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical answers on eight test questions (VER4).

In [ ]:
import csv
import shutil

adapted_results = [pipe.answer(diagram, entry['question'], entry['options'], max_new_tokens=ANSWER_MAX_TOKENS) for entry in diagram_questions]
adapted_diagram = evaluation_report(adapted_results, diagram_correct, sample_kind='synthetic')
for before, after in zip(results, adapted_results, strict=True):
    print({'question': before['question'], 'frozen': before['answer'], 'adapted': after['answer']})
print({'drawn_diagram_accuracy': {'frozen': frozen_diagram['metrics'][0]['value'], 'adapted': adapted_diagram['metrics'][0]['value']}})
with open('outputs/pix2struct_ai2d_answers.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'question', 'options', 'correct_option', 'frozen_answer', 'adapted_answer', 'adapted_choice_index'])
    for before, after, gold in zip(results, adapted_results, diagram_correct, strict=True):
        writer.writerow([diagram_name, before['question'], ' | '.join(before['options']), before['options'][gold], before['answer'], after['answer'], after['choice_index']])

artifact_dir = Path('outputs/pix2struct_ai2d_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'pix2struct_ai2d', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = Pix2StructAI2DPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = [r['answer'] for r in pipe.predict(test_records[:8], max_new_tokens=ANSWER_MAX_TOKENS)]
after = [r['answer'] for r in reloaded.predict(test_records[:8], max_new_tokens=ANSWER_MAX_TOKENS)]
parity = {'identical_answers': sum(a == b for a, b in zip(before, after, strict=True)), 'of': len(before)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_answers'] == parity['of']

weight_entry = next(entry for entry in MANIFEST['files'] if entry['path'] == WEIGHT_FILE)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'total_bytes': snapshot.get('total_bytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'SafeTensors (bfloat16, upcast to float32), digest-verified', 'weight_sha256': weight_entry['sha256']},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'repo': CORPUS_REPO, 'revision': CORPUS_REVISION, 'release': CORPUS_RELEASE, 'license': CORPUS_LICENSE, 'file': CORPUS_FILE, 'sample_questions': SAMPLE_QUESTIONS},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'diagram': {'name': diagram_name, 'size': list(diagram.size), 'rgb_sha256': diagram_sha256, 'questions': diagram_questions, 'correct': diagram_correct}, 'items': [{k: r[k] for k in ('question', 'answer', 'choice_index', 'new_tokens', 'truncated', 'seconds')} for r in results], 'frozen_report': frozen_diagram, 'adapted_items': [{k: r[k] for k in ('question', 'answer', 'choice_index', 'new_tokens', 'truncated')} for r in adapted_results], 'adapted_report': adapted_diagram},
    'comparison': comparison,
    'frozen_beats_chance': frozen_beats_chance,
    'adapted_beats_frozen': adapted_beats_frozen,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'float32', 'source': pipe.source},
}
with open('outputs/pix2struct_ai2d_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The frozen model is an AI2D-trained answerer scored on AI2D questions it was not trained on, beside chance and two baselines that never look at the diagram, and a bounded fine-tuning of the answer decoder's last two blocks on a few hundred more questions is then scored on a diagram-disjoint test split — overall, and separately on lettered-label and text-option questions — with an adapter that reloads to identical answers. That is the claim: the adaptation contract works end to end on a real labelled diagram corpus, and the numbers it produces are read against chance, the baselines and the frozen model rather than in isolation. Whether held-out accuracy rose is recorded as `adapted_beats_frozen`, not assumed.

The test split is about 160 questions on whole diagrams from one seeded draw of one shard, the validation split that picks the epoch is about 80, and accuracy counts an unmatched answer as wrong — so a lower unmatched rate alone can raise it. Because the checkpoint already saw AI2D's training questions, a small or zero gain is the expected outcome and not a failure of the contract; a learning rate that is too high overfits this little data within an epoch, which the validation-based selector reports by keeping epoch 0. So a gain here says the contract works, not that the adapted model reads diagrams better, and it still answers every question — including unanswerable ones — with a fluent option. Fine-tuning on a narrow sample can also erode the model elsewhere; the drawn diagram re-answered in Section 9 is ten questions of evidence about that, not a measurement.

Three things to carry to real data. **Baselines first:** chance, the position prior and the longest option on *your* questions are the numbers to read before any model's, per category. **Leakage:** keep every question on a diagram in one split (the contract does this) and split by source textbook or chapter when your diagrams come from few sources. **Options are part of the input:** the answer depends on the option set (the inference-only smoke answered `leaf` once `root` was removed), so a changed option order or wording is a changed request.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real labelled diagram corpus, validate the demonstrated dataset contract without leakage, execute the inference contract and a bounded fine-tuning, evaluate against chance, two trivial baselines and the frozen model on a diagram-disjoint split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, accuracy on any other diagram population, or production fitness.

**Optional experiments (they do not affect the default path):** raise `LEARNING_RATE` and watch the training loss fall while the validation accuracy drops and the selector keeps an early epoch; set `TRAINABLE_DECODER_LAYERS = 1` and compare the artifact size and the held-out accuracy; shuffle the options of a test question and watch the answer follow them; or bring your own diagrams through BYOD and read the baselines before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/pix2struct-ai2d-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/pix2struct-ai2d-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/pix2struct-ai2d-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model (Google, Apache-2.0): https://huggingface.co/google/pix2struct-ai2d-base
- Upstream code: https://github.com/google-research/pix2struct
- Pix2Struct: Screenshot Parsing as Pretraining for Visual Language Understanding (Lee et al., ICML 2023): https://arxiv.org/abs/2210.03347
- A Diagram Is Worth A Dozen Images — the AI2D dataset (Kembhavi et al., ECCV 2016): https://arxiv.org/abs/1603.07396
- AI2D test questions as mirrored on the Hugging Face Hub: https://huggingface.co/datasets/lmms-lab-encoder/ai2d
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)